# GLOF Eagles 2026 - V14 Final Submission (Inference Only)\nThis notebook performs pure inference using the validated V14 Curriculum_HardMining checkpoints. NO TRAINING CODE IS PRESENT HERE.

In [ ]:
import os\nimport cv2\nimport torch\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport albumentations as A\nfrom albumentations.pytorch import ToTensorV2\nimport timm\nimport segmentation_models_pytorch as smp\n\nDEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nIMG_SIZE = 384\nCATEGORIES = ['Cloud Cover', 'Debris Cover', 'Moraine', 'Snow Cover', 'Terrain Shadow', 'Turbidity']

## 1. Data Loading

In [ ]:
transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(), ToTensorV2()])\n\ndef load_image(path):\n    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)\n    return transform(image=img)['image'].unsqueeze(0).to(DEVICE), img

## 2. Model Loading

In [ ]:
def load_classifier(weights_path):\n    model = timm.create_model('efficientnet_b4', pretrained=False, num_classes=6)\n    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))\n    return model.to(DEVICE).eval()\n\ndef load_segmenter(weights_path):\n    model = smp.UnetPlusPlus('efficientnet-b4', encoder_weights=None, in_channels=3, classes=1)\n    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))\n    return model.to(DEVICE).eval()\n\n# cls_model = load_classifier('cls_weights.pth')\n# seg_model = load_segmenter('model.pth')

## 3. Inference & Visualization

In [ ]:
def infer_and_visualize(image_path, cls_model, seg_model):\n    tensor_img, orig_img = load_image(image_path)\n    with torch.no_grad():\n        # Classification\n        cls_logits = cls_model(tensor_img)\n        pred_class = CATEGORIES[cls_logits.argmax(1).item()]\n        \n        # Segmentation\n        seg_logits = seg_model(tensor_img)\n        mask = (torch.sigmoid(seg_logits) > 0.5).cpu().numpy()[0, 0]\n        \n    mask_resized = cv2.resize(mask.astype(np.float32), (orig_img.shape[1], orig_img.shape[0]), interpolation=cv2.INTER_NEAREST)\n    \n    plt.figure(figsize=(10, 5))\n    plt.subplot(1, 2, 1)\n    plt.title(f'Predicted: {pred_class}')\n    plt.imshow(orig_img)\n    plt.axis('off')\n    \n    plt.subplot(1, 2, 2)\n    plt.title('Segmentation Mask')\n    plt.imshow(mask_resized, cmap='gray')\n    plt.axis('off')\n    plt.show()\n    \n    return pred_class, mask_resized